hfouee


In [0]:
import pyspark.sql.functions as F
import pandas
from datetime import datetime, timedelta
from pyspark.sql.window import Window
import os
from marel.services import ServiceProvider
from marel.databricks import DatabricksProvider
from pyspark.ml.feature import StandardScaler
from pyspark.ml.functions import vector_to_array
import shap
import numpy as np
from pyspark.ml.linalg import Vectors, VectorUDT
from pyspark.sql.types import StructType, StructField

from pyspark.ml.classification import MultilayerPerceptronClassifier
import mlflow

provider = DatabricksProvider()
table_service = provider.table_service

In [0]:
%run /Users/valgerdur.valsdottir@marel.com/LIKX02/df_read_data_prodmedallion

In [0]:
# Read data using the shared function (112 features, no onTime)
days = 30
df_sx_data = df_read_data_prodmedallion(days)

In [0]:
os.environ["MLFLOW_DFS_TMP"] = "/Volumes/teams/sensorx/data-dump/mlflow_tmp"

# Load model (v7: 112 features,
#  15-day horizon)
model = mlflow.spark.load_model("models:/teams.sensorx.mlp_fault_prediction/7")
predictions = model.transform(df_sx_data).cache()

row_count = predictions.count()
print(f"Predictions for 15-day horizon: {row_count:,} rows")

## Output for decision makers

In [0]:
# --- MLP 15-day Model Inference: Daily Ranked Risk Output + SHAP for top 5 ---

mlp_predictions = predictions

# Feature names (112 total)
sensor_cols = [
    "payload_xrayController_filamentCurrent",
    "payload_xrayController_temperature",
    "payload_xrayController_tubeCurrent",
    "payload_xrayController_tubeVoltage",
    "delta_seconds",
]
n_lags = 20
lag_cols = [f"{col}{L}" for L in range(1, n_lags + 1) for col in sensor_cols]
dev_sensors = [
    "payload_xrayController_filamentCurrent",
    "payload_xrayController_temperature",
    "payload_xrayController_tubeCurrent",
]
dev_cols = []
for col in dev_sensors:
    dev_cols.extend([f"{col}_avg_daily", f"{col}_dev_daily"])
feature_names = sensor_cols + lag_cols + dev_cols + ["deviceId_index"]
short_names = [
    n.replace("payload_xrayController_", "")
     .replace("_avg_daily", "_avg")
     .replace("_dev_daily", "_dev")
    for n in feature_names
]

# ── Extract probability column ───────────────────────────────────────────────
mlp_scored = mlp_predictions.select(
    "properties_payloadTimeStamp",
    "properties_deviceID",
    "features",
    F.round(vector_to_array("probability")[1], 4).alias("fault_probability"),
)

# ── Daily p90 score per device ───────────────────────────────────────────────
daily_scores_all = (
    mlp_scored
    .groupBy(
        "properties_deviceID",
        F.to_date("properties_payloadTimeStamp").alias("date")
    )
    .agg(
        F.percentile_approx("fault_probability", 0.9).alias("p90_risk"),
        F.avg("fault_probability").alias("avg_risk"),
        F.count("*").alias("minutes_active")
    )
)

# Filter out devices with insufficient data (<60 readings per day)
# These machines don't have enough data points for a reliable daily risk score
daily_scores = daily_scores_all.filter(F.col("minutes_active") > 60)

excluded = (
    daily_scores_all
    .filter(F.col("minutes_active") <= 60)
    .select("properties_deviceID", "date", "minutes_active")
    .distinct()
)
excluded_count = excluded.select("properties_deviceID").distinct().count()
if excluded_count > 0:
    print(f"\u26a0\ufe0f  {excluded_count} machine(s) excluded due to insufficient data (<60 readings/day):")
    for row in excluded.orderBy("properties_deviceID", "date").collect():
        print(f"   {row['properties_deviceID']}  |  {row['date']}  |  {row['minutes_active']} readings")
    print()

# ── 7-day rolling trend ──────────────────────────────────────────────────────
w_trend = (
    Window
    .partitionBy("properties_deviceID")
    .orderBy("date")
    .rowsBetween(-6, 0)
)

daily_with_trend = (
    daily_scores
    .withColumn("risk_trend_7d", F.round(F.avg("p90_risk").over(w_trend), 4))
)

# ── Rank today's devices by 7-day trend ──────────────────────────────────────
latest_date = daily_scores.agg(F.max("date")).collect()[0][0]
w_rank = Window.orderBy(F.col("risk_trend_7d").desc())

today_ranked = (
    daily_with_trend
    .filter(F.col("date") == latest_date)
    .withColumn("risk_rank", F.rank().over(w_rank))
    .orderBy("risk_rank")
    .select(
        F.col("risk_rank").alias("rank"),
        "properties_deviceID",
        F.col("risk_trend_7d").alias("7d_avg_risk"),
        F.round(F.col("p90_risk"), 4).alias("today_p90"),
    )
)

# ── SHAP for top 5 ───────────────────────────────────────────────────────────
top_5_rows = today_ranked.limit(5).collect()
top_5_ids = [row["properties_deviceID"] for row in top_5_rows]
top_5_ranks = {row["properties_deviceID"]: row["rank"] for row in top_5_rows}
top_5_7d = {row["properties_deviceID"]: row["7d_avg_risk"] for row in top_5_rows}
top_5_p90 = {row["properties_deviceID"]: row["today_p90"] for row in top_5_rows}

# Most recent feature vector per top-5 device
w_latest = Window.partitionBy("properties_deviceID").orderBy(
    F.col("properties_payloadTimeStamp").desc()
)
top_5_spark = (
    mlp_scored
    .filter(F.col("properties_deviceID").isin(top_5_ids))
    .withColumn("rn", F.row_number().over(w_latest))
    .filter(F.col("rn") == 1)
    .drop("rn")
)

top_5_pd = top_5_spark.select(
    "properties_deviceID",
    vector_to_array("features").alias("features_array"),
    "fault_probability"
).toPandas()

top_5_pd["rank_order"] = top_5_pd["properties_deviceID"].map(
    {mid: i for i, mid in enumerate(top_5_ids)}
)
top_5_pd = top_5_pd.sort_values("rank_order").reset_index(drop=True)

X_explain = np.array(top_5_pd["features_array"].tolist())
device_ids = top_5_pd["properties_deviceID"].values

# Background sample for SHAP
background_pd = (
    mlp_predictions
    .select(vector_to_array("features").alias("features_array"))
    .sample(fraction=0.0001, seed=42)
    .limit(50)
    .toPandas()
)
X_background = np.array(background_pd["features_array"].tolist())

schema = StructType([StructField("features", VectorUDT(), True)])

def predict_fn_mlp15(X):
    rows = [(Vectors.dense(x.tolist()),) for x in X]
    df = spark.createDataFrame(rows, schema=schema)
    preds = model.transform(df).select(
        vector_to_array("probability")[1].alias("p")
    ).toPandas()
    return preds["p"].values

explainer = shap.KernelExplainer(predict_fn_mlp15, X_background)
shap_values = explainer.shap_values(X_explain, nsamples=200)

# ── Build final results table ────────────────────────────────────────────────
results_rows = []
for i in range(len(X_explain)):
    sv = shap_values[i]
    top_3_idx = np.argsort(np.abs(sv))[-3:][::-1]
    dev = device_ids[i]

    results_rows.append((
        int(top_5_ranks[dev]),
        dev,
        float(top_5_7d[dev]),
        float(top_5_p90[dev]),
        short_names[top_3_idx[0]],
        short_names[top_3_idx[1]],
        short_names[top_3_idx[2]],
    ))

results_df_15d = spark.createDataFrame(
    results_rows,
    [
        "rank",
        "properties_deviceID",
        "7d_avg_risk",
        "today_p90",
        "top_feature_1",
        "top_feature_2",
        "top_feature_3",
    ]
)

display(results_df_15d)

In [0]:
# All machines ranked by average risk score (no SHAP - fast)
all_machines_avg = (
    mlp_scored
    .groupBy("properties_deviceID")
    .agg(
        F.round(F.avg("fault_probability"), 4).alias("avg_risk"),
        F.round(F.max("fault_probability"), 4).alias("max_risk"),
        F.round(F.percentile_approx("fault_probability", 0.9), 4).alias("p90_risk"),
        F.count("*").alias("data_points")
    )
    .orderBy(F.col("avg_risk").desc())
)

print(f"Total machines scored: {all_machines_avg.count()}")
display(all_machines_avg)